In [ ]:
# ==========================================
# MODELO EXPERIMENTAL: Gaussian Mixture Models (GMM)
# Agrupación Probabilística (Soft Clustering)
# ==========================================

import pandas as pd
import numpy as np
import time
import sys
from sklearn.mixture import GaussianMixture

# 1. Conectar con el módulo evaluador oficial del equipo
sys.path.append('../src')
try:
    from evaluador import evaluar_modelo_recomendacion
except ImportError:
    print("⚠️ Recuerda tener 'evaluador.py' en la carpeta 'src'.")

# 2. Cargar los datos
ruta_csv = "C:/Users/basti/proyecto_cd/data/processed/spotify_dataset_processed.csv"
df = pd.read_csv(ruta_csv)

features = ['danceability', 'energy', 'loudness', 'speechiness', 
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Extraer la matriz matemática
X = df[features].values
print(f"✅ Datos cargados listos para modelado probabilístico.")

In [ ]:
print("📊 Entrenando las 100 distribuciones gaussianas musicales...")

# Entrenamos el modelo. 'diag' optimiza el uso de memoria.
gmm_model = GaussianMixture(n_components=100, covariance_type='diag', random_state=42)

# Asignamos cada canción a la campana de Gauss donde tiene mayor probabilidad de pertenecer
df['cluster_gmm'] = gmm_model.fit_predict(X)

print("✅ ¡Modelo GMM entrenado! El mapa probabilístico ha sido construido.")

In [ ]:
# --- 1. FUNCIÓN DE BÚSQUEDA ---
def buscar_cancion_id(nombre, dataset, artista=None):
    filtro_nombre = dataset['track_name'].str.lower() == nombre.lower()
    resultado = dataset[filtro_nombre]
    if resultado.empty:
        print(f"❌ No se encontró: '{nombre}'")
        return None
    if artista:
        filtro_artista = resultado['artists'].str.contains(artista, case=False, na=False)
        resultado = resultado[filtro_artista]
        if resultado.empty:
            return None
    if len(resultado) > 1:
        print(f"⚠️ Múltiples versiones. Usando la de: {resultado.iloc[0]['artists']}")
    return resultado.iloc[0]['track_id']


# --- 2. MOTOR DE EXPLICABILIDAD ---
def generar_explicacion_acustica(vector_semilla, vector_recomendado, features_list):
    diferencias = np.abs(vector_semilla.values[0] - vector_recomendado.values[0])
    traduccion = {
        'danceability': 'Bailabilidad', 'energy': 'Nivel de Energía', 'loudness': 'Volumen',
        'speechiness': 'Presencia Vocal', 'acousticness': 'Acústica', 'instrumentalness': 'Instrumentalidad',
        'liveness': 'Sensación de Vivo', 'valence': 'Positividad Musical', 'tempo': 'Ritmo'
    }
    dif_dict = {features_list[i]: diferencias[i] for i in range(len(features_list))}
    top_3 = sorted(dif_dict.items(), key=lambda item: item[1])[:3]
    vars_explicacion = [traduccion[var[0]] for var in top_3]
    return f"💡 Recomendada por su similitud en: {vars_explicacion[0]}, {vars_explicacion[1]} y {vars_explicacion[2]}."


# --- 3. MOTOR GMM AUTO-EVALUABLE ---
def recomendar_por_nombre_gmm_y_evaluar(nombre_cancion, dataset, features_list, artista=None, top_k=5):
    # ⏱️ Iniciar cronómetro
    inicio_tiempo = time.time()
    
    # Buscar ID de la semilla
    track_id = buscar_cancion_id(nombre_cancion, dataset, artista)
    if not track_id:
        return None, None
        
    idx_semilla = dataset.index[dataset['track_id'] == track_id].tolist()[0]
    vector_semilla = dataset.iloc[[idx_semilla]][features_list]
    
    nombre_semilla = dataset.iloc[idx_semilla]['track_name']
    artista_semilla = dataset.iloc[idx_semilla]['artists']
    print(f"\n🔍 Explorando la distribución gaussiana para: '{nombre_semilla}' de {artista_semilla}")
    print("-" * 70)

    # Lógica GMM: Buscar el clúster al que pertenece la semilla
    cluster_id = dataset.iloc[idx_semilla]['cluster_gmm']
    
    # Extraer canciones de la misma campana de Gauss (excluyendo la semilla)
    canciones_mismo_cluster = dataset[(dataset['cluster_gmm'] == cluster_id) & (dataset['track_id'] != track_id)]
    
    # Seleccionar recomendaciones
    if len(canciones_mismo_cluster) >= top_k:
        recomendadas = canciones_mismo_cluster.sample(n=top_k, random_state=42)
    else:
        recomendadas = canciones_mismo_cluster
        
    lista_recomendaciones = []
    
    # Procesar y explicar los resultados
    for i, (idx, track_rec) in enumerate(recomendadas.iterrows(), 1):
        vector_rec = dataset.iloc[[idx]][features_list]
        explicacion = generar_explicacion_acustica(vector_semilla, vector_rec, features_list)
        
        print(f"{i}. 🎵 {track_rec['track_name']} - {track_rec['artists']}")
        print(f"   {explicacion}\n")
        
        lista_recomendaciones.append({
            'track_name': track_rec['track_name'],
            'artists': track_rec['artists']
        })
        
    print("-" * 70)
    
    # Extraer vectores para el evaluador
    vectores_recomendados = recomendadas[features_list]
    
    # ⏱️ Detener cronómetro
    fin_tiempo = time.time()
    
    # Auditar con la función central del equipo
    try:
        evaluar_modelo_recomendacion("Gaussian Mixture Models (GMM)", vector_semilla, vectores_recomendados, inicio_tiempo, fin_tiempo)
    except NameError:
        print("⚠️ Evaluador no disponible en memoria.")
        
    return lista_recomendaciones, vectores_recomendados

In [ ]:
def iniciar_app_gmm(df, features_list):
    print("=" * 65)
    print("🎧 SISTEMA DE RECOMENDACIÓN PROBABILÍSTICO (MOTOR GMM) 🎧")
    print("=" * 65)
    
    while True:
        print("\n" + "-" * 65)
        entrada_cancion = input("🎵 Ingresa el nombre de la canción (o 'salir'): ").strip()
        
        if entrada_cancion.lower() == 'salir':
            print("\n¡Gracias por probar el motor probabilístico! Cerrando... 👋")
            break
            
        entrada_artista = input("🎤 Ingresa el artista (Enter para omitir): ").strip()
        entrada_artista = None if entrada_artista == "" else entrada_artista
            
        print("\nProcesando solicitud mediante distribuciones gaussianas...")
        
        # Ejecutar el motor
        recomendar_por_nombre_gmm_y_evaluar(
            nombre_cancion=entrada_cancion, 
            dataset=df, 
            features_list=features_list, 
            artista=entrada_artista
        )

# ¡Arrancar la app!
iniciar_app_gmm(df, features)